In [1]:
import pyarrow.dataset as ds

dataset = ds.dataset(
    # "../output/illustris_skirt.parquet",
    "/urz/gpuscratch/its/doserbd/data/SKIRT_synthetic_images/parquet-v4-128/",
    format="parquet",
    exclude_invalid_files=True,
)
schema = dataset.schema
schema_meta = dataset.schema.metadata

print(dataset.count_rows())
print(schema)

58024
data: list<element: float>
  child 0, element: float
simulation: string
snapshot: int32
subhalo_id: int32
-- schema metadata --
data_shape: '(3, 128, 128)'


In [2]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow.compute as pc

if "image" in dataset.schema.names:
    data_col_name = "image"
elif "data" in dataset.schema.names:
    data_col_name = "data"
else:
    raise ValueError("No data column found in the dataset.")
n_cols = 5
n_rows = 4
page_size = n_cols * n_rows

search_box = widgets.Text(value="", placeholder="Enter subhalo_id", description="Subhalo ID:", continuous_update=False)
slider = widgets.IntSlider(value=0, min=0, max=0, description="Page:")
btn_prev = widgets.Button(description="-", layout=widgets.Layout(width="40px"))
btn_next = widgets.Button(description="+", layout=widgets.Layout(width="40px"))
out = widgets.Output()

_cached_table = None


def get_filtered_table():
    global _cached_table
    sid_text = search_box.value.strip()
    columns = [data_col_name, "simulation", "snapshot", "subhalo_id"]
    if sid_text:
        try:
            filt = pc.equal(ds.field("subhalo_id"), int(sid_text))
            _cached_table = dataset.to_table(columns=columns, filter=filt)
        except ValueError:
            _cached_table = dataset.to_table(columns=columns)
    else:
        _cached_table = dataset.to_table(columns=columns)
    return _cached_table


def update_slider(*args):
    table = get_filtered_table()
    n_pages = max(1, (len(table) + page_size - 1) // page_size)
    slider.max = n_pages - 1
    slider.value = 0
    show_page(0)


def show_page(page):
    if _cached_table is None:
        return
    table = _cached_table.slice(page * page_size, page_size).to_pydict()
    batch = table[data_col_name]
    simulations = table["simulation"]
    snapshots = table["snapshot"]
    subhalo_ids = table["subhalo_id"]

    key = b"data_shape"
    data_shape = None
    if key in schema_meta:
        parts = schema_meta[key].decode().strip("()").split(",")
        data_shape = tuple(int(p) for p in parts if p.strip())

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 20))
        for ax, img_channels, sim, snap, sid in zip(axes.flatten(), batch, simulations, snapshots, subhalo_ids):
            if data_col_name == "image":
                data = np.stack([np.stack(ch) for ch in img_channels]).transpose(1, 2, 0) * 255
            else:
                data = np.array(img_channels).reshape(data_shape).transpose(1, 2, 0) * 255
            image = Image.fromarray(data.astype(np.uint8), "RGB")
            ax.imshow(image)
            ax.set_title(f"{sim} / {snap} / {sid}", fontsize=14)
            ax.axis("off")
        for ax in axes.flatten()[len(batch) :]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


def on_prev(b):
    if slider.value > slider.min:
        slider.value -= 1


def on_next(b):
    if slider.value < slider.max:
        slider.value += 1


search_box.observe(update_slider, names="value")
slider.observe(lambda change: show_page(change["new"]), names="value")
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
display(search_box, widgets.HBox([btn_prev, slider, btn_next]), out)
update_slider()


Text(value='', continuous_update=False, description='Subhalo ID:', placeholder='Enter subhalo_id')

Output()